# CLAP

In [ ]:
import torch
import laion_clap

model = laion_clap.CLAP_Module(enable_fusion=False, amodel="HTSAT-tiny")
model.load_ckpt("../630k-audioset-best.pt")
model = model.cuda().eval()


In [ ]:

# [B, T], mono waveform, expected around 48 kHz for LAION CLAP

import torchaudio
st = time.time()

audio = torch.randn(32, 44100 * 10, device="cuda")
audio = torchaudio.functional.resample(audio, orig_freq=44100, new_freq=48000)

import time 

with torch.inference_mode():
    z = model.get_audio_embedding_from_data(
        x=audio,
        use_tensor=True,
    )
end = time.time()
print("Time taken for embedding:", end - st)
print(z.shape)

# PHASE LOSS 

In [ ]:
import torch


def wrap_phase(x):
    """Wrap angles to [-pi, pi]."""
    return torch.atan2(torch.sin(x), torch.cos(x))


def phase_derivative_losses(source, target):
    """
    source, target: complex STFT tensors
        shape [..., F, T]

    Returns:
        loss_if: phase-derivative loss along time
        loss_gd: phase-derivative loss along frequency
    """

    source_phase = torch.angle(source)
    target_phase = torch.angle(target)

    # Instantaneous frequency: phase evolution over time
    source_if = torch.diff(source_phase, dim=-1)
    target_if = torch.diff(target_phase, dim=-1)

    if_error = wrap_phase(source_if - target_if)
    loss_if = if_error.abs().mean()

    # Group delay: phase evolution over frequency
    source_gd = torch.diff(source_phase, dim=-2)
    target_gd = torch.diff(target_phase, dim=-2)

    gd_error = wrap_phase(source_gd - target_gd)
    loss_gd = gd_error.abs().mean()

    return loss_if, loss_gd

In [ ]:
import math
import torch
import torchaudio.functional as AF


def _biquad_coeffs_high_shelf(
    sample_rate,
    f0=1681.974,
    gain_db=4.0,
    q=0.7071,
    device=None,
    dtype=None,
):
    A = 10 ** (gain_db / 40.0)
    w0 = 2 * math.pi * f0 / sample_rate
    alpha = math.sin(w0) / (2 * q)
    cos_w0 = math.cos(w0)
    sqrt_A = math.sqrt(A)

    b0 = A * ((A + 1) + (A - 1) * cos_w0 + 2 * sqrt_A * alpha)
    b1 = -2 * A * ((A - 1) + (A + 1) * cos_w0)
    b2 = A * ((A + 1) + (A - 1) * cos_w0 - 2 * sqrt_A * alpha)

    a0 = (A + 1) - (A - 1) * cos_w0 + 2 * sqrt_A * alpha
    a1 = 2 * ((A - 1) - (A + 1) * cos_w0)
    a2 = (A + 1) - (A - 1) * cos_w0 - 2 * sqrt_A * alpha

    b = torch.tensor([b0, b1, b2], device=device, dtype=dtype) / a0
    a = torch.tensor([1.0, a1 / a0, a2 / a0], device=device, dtype=dtype)

    return b, a


def _biquad_coeffs_highpass(
    sample_rate,
    f0=38.135,
    q=0.5,
    device=None,
    dtype=None,
):
    w0 = 2 * math.pi * f0 / sample_rate
    alpha = math.sin(w0) / (2 * q)
    cos_w0 = math.cos(w0)

    b0 = (1 + cos_w0) / 2
    b1 = -(1 + cos_w0)
    b2 = (1 + cos_w0) / 2

    a0 = 1 + alpha
    a1 = -2 * cos_w0
    a2 = 1 - alpha

    b = torch.tensor([b0, b1, b2], device=device, dtype=dtype) / a0
    a = torch.tensor([1.0, a1 / a0, a2 / a0], device=device, dtype=dtype)

    return b, a


def k_weighting(x, sample_rate):
    """
    x: waveform [..., T]

    Returns K-weighted waveform with the same shape.
    """

    dtype = x.dtype
    device = x.device

    # Stage 1: ~+4 dB high-frequency shelf
    b, a = _biquad_coeffs_high_shelf(
        sample_rate,
        device=device,
        dtype=dtype,
    )
    x = AF.lfilter(x, a, b)

    # Stage 2: low-frequency roll-off / RLB high-pass
    b, a = _biquad_coeffs_highpass(
        sample_rate,
        device=device,
        dtype=dtype,
    )
    x = AF.lfilter(x, a, b)

    return x

In [ ]:
import torchaudio
audio, sr = torchaudio.load("/data/nils/repos/AFTER/patchs/data/audio_files/break0.wav")


In [ ]:
import torch
import matplotlib.pyplot as plt


def plot_spectrograms_and_energy(audio_a, audio_b, n_fft=1024, sr=44100):
    def compute(audio):
        audio = torch.as_tensor(audio).float().squeeze().cpu()

        hop = n_fft // 4
        window = torch.hann_window(n_fft)

        spec = torch.stft(
            audio,
            n_fft=n_fft,
            hop_length=hop,
            win_length=n_fft,
            window=window,
            return_complex=True,
            normalized=False
        )

        mag = spec.abs()
        power = mag.square()

        spec_db = 20 * torch.log10(mag + 1e-8)
        energy_db = 10 * torch.log10(power.mean(dim=-1) + 1e-12)

        duration = len(audio) / sr
        freqs = torch.linspace(0, sr / 2, spec.shape[0])

        return spec_db.numpy(), energy_db.numpy(), freqs.numpy(), duration

    spec_a, energy_a, freqs, duration_a = compute(audio_a)
    spec_b, energy_b, _, duration_b = compute(audio_b)

    vmax = max(spec_a.max(), spec_b.max())
    vmin = vmax - 80

    fig = plt.figure(figsize=(14, 9))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1])

    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_energy = fig.add_subplot(gs[1, :])

    im_a = ax_a.imshow(
        spec_a,
        origin="lower",
        aspect="auto",
        extent=[0, duration_a, 0, sr / 2],
        vmin=vmin,
        vmax=vmax,
    )
    ax_a.set_title("Audio A")
    ax_a.set_xlabel("Time (s)")
    ax_a.set_ylabel("Frequency (Hz)")

    im_b = ax_b.imshow(
        spec_b,
        origin="lower",
        aspect="auto",
        extent=[0, duration_b, 0, sr / 2],
        vmin=vmin,
        vmax=vmax,
    )
    ax_b.set_title("Audio B")
    ax_b.set_xlabel("Time (s)")
    ax_b.set_ylabel("Frequency (Hz)")
    
    
    energy_a = torch.as_tensor(energy_a)
    energy_b = torch.as_tensor(energy_b)

    delta_db = energy_b - energy_a

    plt.figure(figsize=(8, 4))
    ax_energy.semilogx(freqs[1:], delta_db[1:])

    # plt.axhline(0, linewidth=0.8)
    # plt.xlim(10, sr / 2)
    # plt.xlabel("Frequency (Hz)")
    # plt.ylabel("B - A (dB)")
    # plt.grid(True)
    # plt.show()

    # ax_energy.plot(freqs, energy_a, label="Audio A")
    # ax_energy.plot(freqs, energy_b, label="Audio B")
    # ax_energy.set_xlabel("Frequency (Hz)")
    # ax_energy.set_ylabel("Mean spectral energy (dB)")
    # ax_energy.set_ylim(-20,30)
    # ax_energy.legend()

    fig.colorbar(im_b, ax=[ax_a, ax_b], label="Magnitude (dB)")

    plt.show()

In [ ]:
audio, sr = torchaudio.load("/data/nils/repos/AFTER/patchs/data/audio_files/break0.wav")
audio_transformed = k_weighting(audio, sr)
plot_spectrograms_and_energy(audio_a=audio.squeeze(), audio_b=audio_transformed.squeeze(), n_fft=4096, sr=44100)

In [ ]:
import torch
import matplotlib.pyplot as plt


def measure_filter_response(filter_fn, sr=44100, n=262144):
    x = torch.zeros(n)
    x[0] = 1.0

    y = filter_fn(x)

    H = torch.fft.rfft(y)
    freqs = torch.fft.rfftfreq(n, 1 / sr)

    response_db = 20 * torch.log10(H.abs() + 1e-12)

    plt.figure(figsize=(9, 4))
    plt.semilogx(freqs[1:], response_db[1:])
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Gain (dB)")
    plt.xlim(10, sr / 2)
    plt.grid(True)
    plt.show()
    
    
measure_filter_response(
    lambda x: k_weighting(x, 44100),
    sr=44100,
)

In [ ]:
import math
import time
import torch
import torchaudio
import torchaudio.functional as AF
import pyrubberband as pyrb

from IPython.display import Audio, display

from after.autoencoder.transforms import TimeStretch


# ------------------------------------------------------------
# Load audio
# ------------------------------------------------------------

audio, sr = torchaudio.load(
    "/data/nils/repos/AFTER/patchs/data/audio_files/break0.wav"
)

y = audio.squeeze(0)[..., :131072]   # torch [T]
y_np = y.numpy()


# ------------------------------------------------------------
# CUDA phase-vocoder implementation
# ------------------------------------------------------------

@torch.inference_mode()
def torch_time_stretch(x, rate=1.1, n_fft=2048, hop_length=None):
    """
    x: [T] or [B, T] torch tensor on CUDA
    rate > 1 -> faster / shorter
    """
    if hop_length is None:
        hop_length = n_fft // 4

    window = torch.hann_window(
        n_fft,
        device=x.device,
        dtype=x.dtype,
    )

    X = torch.stft(
        x,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=n_fft,
        window=window,
        return_complex=True,
    )

    phase_advance = torch.linspace(
        0,
        math.pi * hop_length,
        X.shape[-2],
        device=x.device,
        dtype=x.dtype,
    )[..., None]

    Y = AF.phase_vocoder(
        X,
        rate=rate,
        phase_advance=phase_advance,
    )

    target_length = round(x.shape[-1] / rate)

    y_out = torch.istft(
        Y,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=n_fft,
        window=window,
        length=target_length,
    )

    return y_out


rate = 1.1


# ------------------------------------------------------------
# Rubber Band
# ------------------------------------------------------------

# warm-up
_ = pyrb.time_stretch(
    y_np,
    sr=sr,
    rate=rate,
)

st = time.time()

y_rubberband = pyrb.time_stretch(
    y_np,
    sr=sr,
    rate=rate,
)

rubberband_time = time.time() - st

print(f"Rubber Band: {rubberband_time:.4f} s")


# ------------------------------------------------------------
# AFTER TimeStretch
# ------------------------------------------------------------

ts = TimeStretch(p=1.)

# warm-up
_ = ts(y_np, sr)

st = time.time()

y_after = ts(y_np, sr)

after_time = time.time() - st

print(f"AFTER TimeStretch: {after_time:.4f} s")


# ------------------------------------------------------------
# Torch CUDA phase vocoder
# ------------------------------------------------------------

device = "cuda"

y_cuda = y.to(device)

# warm-up
_ = torch_time_stretch(
    y_cuda,
    rate=rate,
    n_fft=2048,
)
torch.cuda.synchronize()

st = time.time()

y_torch = torch_time_stretch(
    y_cuda,
    rate=rate,
    n_fft=2048,
)

torch.cuda.synchronize()

torch_time = time.time() - st

y_torch_cpu = y_torch.cpu().numpy()

print(f"Torch CUDA PV: {torch_time:.4f} s")


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("Input duration       :", len(y_np) / sr)
print("Rubber Band duration :", len(y_rubberband) / sr)
print("AFTER duration       :", len(y_after) / sr)
print("Torch CUDA duration  :", len(y_torch_cpu) / sr)


# ------------------------------------------------------------
# Listen
# ------------------------------------------------------------

print("\nOriginal")
display(Audio(y_np, rate=sr))

print("Rubber Band")
display(Audio(y_rubberband, rate=sr))

print("AFTER TimeStretch")
display(Audio(y_after, rate=sr))

print("Torch CUDA phase vocoder")
display(Audio(y_torch_cpu, rate=sr))

In [ ]:
import numpy as np

latent = torch.randn((1, 64, y.shape[-1]//256))

base = y.shape[-1]/256
print("base : ", base)



# Select k values based on min and max value accepted (default is 0.8/1.2)
k = np.arange(-20, 20)
rates = 1 + k*256/y.shape[-1]

rate = rates[12]

ts = TimeStretch(p=1.,min_rate = 1/rate, max_rate=1/rate, leave_length_unchanged=False)

# warm-up
yout = ts(y.numpy(), sr)


# Check that we still have yout%256 = 0 

print("Target ratio", rate, "Out Shape : ", yout.shape)
print("Div 256 : ", yout.shape[-1]%256, yout.shape[-1]//256)

print("Out ratio : ", yout.shape[-1]/ y.shape[-1])

print(latent.shape)

z_up = torch.nn.functional.interpolate(
    latent,
    size=yout.shape[-1]//256,
    mode="linear",
    align_corners=False,
)

print(z_up.shape)
